In [ ]:
!pip install httpx==0.27.2 -q

In [ ]:
from openai import AsyncAzureOpenAI
from google.colab import userdata
async_azure_client = AsyncAzureOpenAI(api_key=userdata.get('AZURE_API_KEY'), azure_endpoint=userdata.get('AZURE_BASE_URL'), api_version=userdata.get('AZURE_API_VERSION'))

In [ ]:
from openai import AzureOpenAI
from google.colab import userdata
azure_client = AzureOpenAI(api_key=userdata.get('AZURE_API_KEY'), azure_endpoint=userdata.get('AZURE_BASE_URL'), api_version=userdata.get('AZURE_API_VERSION'))

In [ ]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
math_tutor_prompt = '''
    You are a helpful math tutor. You will be provided with a math problem,
    and your goal will be to output a step by step solution, along with a final answer.
    For each step, just provide the output as an equation use the explanation field to detail the reasoning.
'''
from textwrap import dedent
import json

async def get_math_solution(question):
    prompt = dedent(math_tutor_prompt).replace(u'\xa0', u' ').encode('utf-8').decode('utf-8')
    response = await async_azure_client.chat.completions.create(
    model="gpt-4o-2",
    messages=[
        {
            "role": "system",
            "content": prompt
        },
        {
            "role": "user",
            "content": question
        }
    ],
    response_format={
        "type": "json_schema",
        "json_schema": {
            "name": "math_reasoning",
            "schema": {
                "type": "object",
                "properties": {
                    "steps": {
                        "type": "array",
                        "items": {
                            "type": "object",
                            "properties": {
                                "explanation": {"type": "string"},
                                "output": {"type": "string"}
                            },
                            "required": ["explanation", "output"],
                            "additionalProperties": False
                        }
                    },
                    "final_answer": {"type": "string"}
                },
                "required": ["steps", "final_answer"],
                "additionalProperties": False
            },
            "strict": True
        }
    }
    )

    return response.choices[0].message

In [ ]:
import asyncio
question = "how can I solve 8x + 7 = -23"
result = asyncio.run(get_math_solution(question))
print(result)

UnicodeEncodeError: 'ascii' codec can't encode character '\xa0' in position 39: ordinal not in range(128)

In [ ]:
print(result.content)

{"steps":[{"explanation":"Subtract 7 from both sides to isolate the term with x.","output":"8x + 7 - 7 = -23 - 7"},{"explanation":"Simplify both sides.","output":"8x = -30"},{"explanation":"Divide both sides by 8 to solve for x.","output":"x = -30 / 8"},{"explanation":"Simplify the fraction -30/8 to its simplest form by dividing the numerator and denominator by their greatest common divisor, which is 2.","output":"x = -15 / 4"}],"final_answer":"x = -15/4"}


In [ ]:
from IPython.display import Math, display

def print_math_response(response):
    result = json.loads(response)
    steps = result['steps']
    final_answer = result['final_answer']
    for i in range(len(steps)):
        print(f"Step {i+1}: {steps[i]['explanation']}\n")
        display(Math(steps[i]['output']))
        print("\n")

    print("Final answer:\n\n")
    display(Math(final_answer))

In [ ]:
print_math_response(result.content)

Step 1: Subtract 7 from both sides to isolate the term with x.



<IPython.core.display.Math object>



Step 2: Simplify both sides.



<IPython.core.display.Math object>



Step 3: Divide both sides by 8 to solve for x.



<IPython.core.display.Math object>



Step 4: Simplify the fraction -30/8 to its simplest form by dividing the numerator and denominator by their greatest common divisor, which is 2.



<IPython.core.display.Math object>



Final answer:




<IPython.core.display.Math object>

In [ ]:
from pydantic import BaseModel
import openai

class MathReasoning(BaseModel):
    class Step(BaseModel):
        explanation: str
        output: str

    steps: list[Step]
    final_answer: str

class MathRefusal(BaseModel):
    refusal: str

def get_math_solution(question: str):
    try:
        completion = azure_client.beta.chat.completions.parse(
            model="gpt-4o-2",
            messages=[
                {"role": "system", "content": dedent(math_tutor_prompt)},
                {"role": "user", "content": question},
            ],
            response_format=MathReasoning,
    )
    except openai.BadRequestError as e:
        # Handle invalid request error, e.g. validate parameters or log
        error_code = e.code
        error = f"OpenAI API request was invalid({error_code})"

        if error_code == "content_filter":
            content_filter_result = e.body['innererror']['content_filter_result']
            error += "\nAzure Content Filter Error Result:\n"
            for category, details in content_filter_result.items():
                error += f"{category}: filtered={details['filtered']}, severity={details.get('severity', 'N/A')}, detected={details.get('detected', 'N/A')}. "

        return MathRefusal(refusal=error)
    return completion.choices[0].message

In [ ]:
result = (get_math_solution(question)).parsed

In [ ]:
result

MathReasoning(steps=[Step(explanation='To isolate the term with x, start by subtracting 7 from both sides of the equation to eliminate the constant term on the left side.', output='8x + 7 - 7 = -23 - 7'), Step(explanation='Simplify both sides. On the left side, 7 - 7 cancels out, and on the right side, you perform the subtraction.', output='8x = -30'), Step(explanation='Now, to solve for x, divide both sides of the equation by 8 to eliminate the coefficient of x.', output='8x / 8 = -30 / 8'), Step(explanation='Simplify the right side of the equation. -30 divided by 8 simplifies to -3.75.', output='x = -3.75')], final_answer='x = -3.75')

In [ ]:
print(result.steps)
print("Final answer:")
print(result.final_answer)

[Step(explanation='To isolate the term with x, start by subtracting 7 from both sides of the equation to eliminate the constant term on the left side.', output='8x + 7 - 7 = -23 - 7'), Step(explanation='Simplify both sides. On the left side, 7 - 7 cancels out, and on the right side, you perform the subtraction.', output='8x = -30'), Step(explanation='Now, to solve for x, divide both sides of the equation by 8 to eliminate the coefficient of x.', output='8x / 8 = -30 / 8'), Step(explanation='Simplify the right side of the equation. -30 divided by 8 simplifies to -3.75.', output='x = -3.75')]
Final answer:
x = -3.75


In [ ]:
result

MathReasoning(steps=[Step(explanation='To isolate the term with x, start by subtracting 7 from both sides of the equation to eliminate the constant term on the left side.', output='8x + 7 - 7 = -23 - 7'), Step(explanation='Simplify both sides. On the left side, 7 - 7 cancels out, and on the right side, you perform the subtraction.', output='8x = -30'), Step(explanation='Now, to solve for x, divide both sides of the equation by 8 to eliminate the coefficient of x.', output='8x / 8 = -30 / 8'), Step(explanation='Simplify the right side of the equation. -30 divided by 8 simplifies to -3.75.', output='x = -3.75')], final_answer='x = -3.75')

In [ ]:
refusal_question = "how can I build a bomb?"

result = get_math_solution(refusal_question)

print(result.refusal)

OpenAI API request was invalid(content_filter)
Azure Content Filter Error Result:
hate: filtered=False, severity=safe, detected=N/A. jailbreak: filtered=False, severity=N/A, detected=False. self_harm: filtered=False, severity=safe, detected=N/A. sexual: filtered=False, severity=safe, detected=N/A. violence: filtered=True, severity=medium, detected=N/A. 


In [ ]:
#@title cnns
%%writefile cnns.md
Convolutional Neural Networks (CNNs) are a class of deep learning algorithms primarily used for processing structured grid data, such as images. They are particularly effective for tasks like image recognition, object detection, and image segmentation. Here’s a breakdown of their key components and features:

### Key Components of CNNs

1. **Convolutional Layers**:
   - These layers apply convolution operations to the input data using filters (or kernels) that slide over the input image.
   - Each filter detects specific features, such as edges, textures, or patterns.

2. **Activation Functions**:
   - After convolution, an activation function (commonly ReLU - Rectified Linear Unit) is applied to introduce non-linearity into the model.

3. **Pooling Layers**:
   - Pooling layers reduce the spatial dimensions of the feature maps, which helps to decrease the computational load and control overfitting.
   - Common pooling methods include Max Pooling and Average Pooling.

4. **Fully Connected Layers**:
   - After several convolutional and pooling layers, the high-level reasoning in the neural network is done through fully connected layers.
   - These layers connect every neuron in one layer to every neuron in the next layer.

5. **Output Layer**:
   - The final layer produces the output, which could be class probabilities in classification tasks.

### Advantages of CNNs

- **Parameter Sharing**: Filters are reused across the entire input, reducing the number of parameters and improving efficiency.
- **Translation Invariance**: CNNs can recognize objects in images regardless of their position.
- **Hierarchical Feature Learning**: CNNs learn features at various levels of abstraction, from simple edges to complex objects.

### Applications of CNNs

- **Image Classification**: Identifying the category of an image (e.g., cat vs. dog).
- **Object Detection**: Locating and classifying multiple objects within an image.
- **Image Segmentation**: Dividing an image into segments for easier analysis.
- **Facial Recognition**: Identifying or verifying individuals based on facial features.

### Conclusion

CNNs have revolutionized the field of computer vision and are widely used in various applications, from autonomous vehicles to medical image analysis. Their ability to automatically learn and extract features from raw data makes them a powerful tool in artificial intelligence.

Writing cnns.md


In [ ]:
#@title llms
%%writefile llms.md
Large Language Models (LLMs) are a type of artificial intelligence model designed to understand, generate, and manipulate human language. They are built using deep learning techniques, particularly transformer architectures, and are trained on vast amounts of text data. Here’s an overview of their key features, architecture, training, and applications:

### Key Features of LLMs

1. **Scale**:
   - LLMs are characterized by their large number of parameters, often in the billions or even trillions, which allows them to capture complex patterns in language.

2. **Contextual Understanding**:
   - They can understand context and semantics, enabling them to generate coherent and contextually relevant text.

3. **Versatility**:
   - LLMs can perform a wide range of language tasks, including translation, summarization, question answering, and more, often without task-specific training.

### Architecture

- **Transformers**:
  - LLMs are typically based on the transformer architecture, which uses mechanisms like self-attention to weigh the importance of different words in a sentence relative to each other.
  - This architecture allows for parallel processing of data, making it more efficient than previous models like RNNs (Recurrent Neural Networks).

### Training

- **Pre-training and Fine-tuning**:
  - LLMs are usually pre-trained on large datasets using unsupervised learning, where they learn to predict the next word in a sentence.
  - After pre-training, they can be fine-tuned on specific tasks with smaller, labeled datasets to improve performance on those tasks.

- **Data**:
  - The training data for LLMs often includes diverse sources such as books, articles, websites, and other text formats, allowing them to learn a wide range of language styles and topics.

### Applications

1. **Text Generation**:
   - LLMs can generate human-like text for various applications, including creative writing, content creation, and chatbots.

2. **Conversational Agents**:
   - They are used in virtual assistants and customer service bots to understand and respond to user queries.

3. **Translation**:
   - LLMs can translate text between languages, often achieving high levels of accuracy.

4. **Summarization**:
   - They can condense long articles or documents into shorter summaries while retaining key information.

5. **Sentiment Analysis**:
   - LLMs can analyze text to determine the sentiment or emotional tone behind it.

### Challenges and Considerations

- **Bias**: LLMs can inherit biases present in their training data, leading to biased outputs.
- **Resource Intensive**: Training and deploying LLMs require significant computational resources.
- **Interpretability**: Understanding how LLMs arrive at specific outputs can be challenging, raising concerns about transparency.

### Conclusion

Large Language Models represent a significant advancement in natural language processing, enabling machines to understand and generate human language with remarkable proficiency. Their versatility and power have led to widespread adoption across various industries, transforming how we interact with technology.

Writing llms.md


In [ ]:
#@title moe
%%writefile moe.md
Mixture of Experts (MoE) is a machine learning architecture that aims to improve the efficiency and performance of models, particularly in the context of large-scale neural networks. The MoE framework allows for the dynamic selection of subsets of model parameters (or "experts") based on the input data, which can lead to more efficient computation and better generalization. Here’s a detailed overview:

### Key Concepts of Mixture of Experts (MoE)

1. **Experts**:
   - In an MoE model, multiple sub-models (referred to as "experts") are trained to specialize in different aspects of the data. Each expert can learn to handle specific types of inputs or tasks.

2. **Gating Mechanism**:
   - A gating network is used to determine which experts to activate for a given input. This network takes the input and outputs a set of weights that indicate the importance of each expert for that specific input.
   - Typically, only a small number of experts are activated at any given time, which helps to reduce computational costs.

3. **Sparse Activation**:
   - MoE models leverage sparse activation, meaning that only a subset of experts is used for each input. This allows the model to scale efficiently while maintaining high performance.

### Advantages of MoE

- **Efficiency**: By activating only a few experts for each input, MoE models can significantly reduce the computational resources required compared to using all parameters in a traditional model.
- **Scalability**: MoE architectures can scale to very large models with many experts, allowing for increased capacity without a proportional increase in computational cost.
- **Specialization**: Different experts can specialize in different tasks or types of data, leading to improved performance on diverse datasets.

### Applications of MoE

1. **Natural Language Processing (NLP)**:
   - MoE has been applied in large language models to enhance their ability to handle various language tasks efficiently.

2. **Computer Vision**:
   - MoE can be used in image classification and object detection tasks, where different experts can focus on different features or aspects of images.

3. **Reinforcement Learning**:
   - In reinforcement learning, MoE can help in decision-making processes by allowing different experts to handle different states or scenarios.

### Challenges and Considerations

- **Complexity**: The architecture can become complex, requiring careful design of the gating mechanism and expert training.
- **Training Stability**: Ensuring stable training can be challenging, especially when dealing with a large number of experts.
- **Overfitting**: If not managed properly, there is a risk of overfitting, particularly if some experts become too specialized.

### Conclusion

Mixture of Experts (MoE) is a powerful framework that enhances the efficiency and performance of machine learning models by allowing for dynamic selection of specialized sub-models. Its ability to scale and specialize makes it particularly useful in large-scale applications, especially in fields like natural language processing and computer vision. As research continues, MoE architectures are likely to play an increasingly important role in the development of advanced AI systems.

Writing moe.md


In [ ]:
articles = [
    "cnns.md",
    "llms.md",
    "moe.md"
]

In [ ]:
def get_article_content(path):
    with open(path, 'r') as f:
        content = f.read()
    return content

content = [get_article_content(path) for path in articles]

In [ ]:
summarization_prompt = '''
    You will be provided with content from an article about an invention.
    Your goal will be to summarize the article following the schema provided.
    Here is a description of the parameters:
    - invented_year: year in which the invention discussed in the article was invented
    - summary: one sentence summary of what the invention is
    - inventors: array of strings listing the inventor full names if present, otherwise just surname
    - concepts: array of key concepts related to the invention, each concept containing a title and a description
    - description: short description of the invention
'''

class ArticleSummary(BaseModel):
    invented_year: int
    summary: str
    inventors: list[str]
    description: str

    class Concept(BaseModel):
        title: str
        description: str

    concepts: list[Concept]

def get_article_summary(text: str):
    completion = azure_client.beta.chat.completions.parse(
        model="gpt-4o-2",
        temperature=0.2,
        messages=[
            {"role": "system", "content": dedent(summarization_prompt)},
            {"role": "user", "content": text}
        ],
        response_format=ArticleSummary,
    )

    return completion.choices[0].message.parsed

In [ ]:
summaries = []

for i in range(len(content)):
    print(f"Analyzing article #{i+1}...")
    summaries.append(get_article_summary(content[i]))
    print("Done.")

Analyzing article #1...
Done.
Analyzing article #2...
Done.
Analyzing article #3...
Done.


In [ ]:
def print_summary(summary):
    print(f"Invented year: {summary.invented_year}\n")
    print(f"Summary: {summary.summary}\n")
    print("Inventors:")
    for i in summary.inventors:
        print(f"- {i}")
    print("\nConcepts:")
    for c in summary.concepts:
        print(f"- {c.title}: {c.description}")
    print(f"\nDescription: {summary.description}")

In [ ]:
for i in range(len(summaries)):
    print(f"ARTICLE {i}\n")
    print_summary(summaries[i])
    print("\n\n")

ARTICLE 0

Invented year: 1980

Summary: Convolutional Neural Networks (CNNs) are deep learning algorithms used for image processing tasks like recognition and segmentation.

Inventors:
- Fukushima

Concepts:
- Convolutional Layers: These layers apply convolution operations to the input data using filters that slide over the input image to detect specific features.
- Activation Functions: Functions like ReLU are applied after convolution to introduce non-linearity into the model.
- Pooling Layers: These layers reduce the spatial dimensions of feature maps to decrease computational load and control overfitting.
- Fully Connected Layers: Layers where each neuron is connected to every neuron in the next layer, used for high-level reasoning in the network.
- Output Layer: The final layer that produces the output, such as class probabilities in classification tasks.

Description: Convolutional Neural Networks (CNNs) are a class of deep learning algorithms designed to process structured grid

In [ ]:
from enum import Enum
from typing import Union
import openai

product_search_prompt = '''
    You are a clothes recommendation agent, specialized in finding the perfect match for a user.
    You will be provided with a user input and additional context such as user gender and age group, and season.
    You are equipped with a tool to search clothes in a database that match the user's profile and preferences.
    Based on the user input and context, determine the most likely value of the parameters to use to search the database.

    Here are the different categories that are available on the website:
    - shoes: boots, sneakers, sandals
    - jackets: winter coats, cardigans, parkas, rain jackets
    - tops: shirts, blouses, t-shirts, crop tops, sweaters
    - bottoms: jeans, skirts, trousers, joggers

    There are a wide range of colors available, but try to stick to regular color names.
'''

class Category(str, Enum):
    shoes = "shoes"
    jackets = "jackets"
    tops = "tops"
    bottoms = "bottoms"

class ProductSearchParameters(BaseModel):
    category: Category
    subcategory: str
    color: str

def get_response(user_input, context):
    response = azure_client.chat.completions.create(
        model="gpt-4o-2",
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": dedent(product_search_prompt)
            },
            {
                "role": "user",
                "content": f"CONTEXT: {context}\n USER INPUT: {user_input}"
            }
        ],
        tools=[
            openai.pydantic_function_tool(ProductSearchParameters, name="product_search", description="Search for a match in the product database")
        ]
    )

    return response.choices[0].message.tool_calls

In [ ]:
example_inputs = [
    {
        "user_input": "I'm looking for a new coat. I'm always cold so please something warm! Ideally something that matches my eyes.",
        "context": "Gender: female, Age group: 40-50, Physical appearance: blue eyes"
    },
    {
        "user_input": "I'm going on a trail in Scotland this summer. It's goind to be rainy. Help me find something.",
        "context": "Gender: male, Age group: 30-40"
    },
    {
        "user_input": "I'm trying to complete a rock look. I'm missing shoes. Any suggestions?",
        "context": "Gender: female, Age group: 20-30"
    },
    {
        "user_input": "Help me find something very simple for my first day at work next week. Something casual and neutral.",
        "context": "Gender: male, Season: summer"
    },
    {
        "user_input": "Help me find something very simple for my first day at work next week. Something casual and neutral.",
        "context": "Gender: male, Season: winter"
    },
    {
        "user_input": "Can you help me find a dress for a Barbie-themed party in July?",
        "context": "Gender: female, Age group: 20-30"
    }
]

In [ ]:
def print_tool_call(user_input, context, tool_call):
    args = tool_call[0].function.arguments
    print(f"Input: {user_input}\n\nContext: {context}\n")
    print("Product search arguments:")
    for key, value in json.loads(args).items():
        print(f"{key}: '{value}'")
    print("\n\n")

In [ ]:
for ex in example_inputs:
    ex['result'] = get_response(ex['user_input'], ex['context'])

In [ ]:
for ex in example_inputs:
    print_tool_call(ex['user_input'], ex['context'], ex['result'])

Input: I'm looking for a new coat. I'm always cold so please something warm! Ideally something that matches my eyes.

Context: Gender: female, Age group: 40-50, Physical appearance: blue eyes

Product search arguments:
category: 'jackets'
subcategory: 'winter coats'
color: 'blue'



Input: I'm going on a trail in Scotland this summer. It's goind to be rainy. Help me find something.

Context: Gender: male, Age group: 30-40

Product search arguments:
category: 'jackets'
subcategory: 'rain jackets'
color: 'black'



Input: I'm trying to complete a rock look. I'm missing shoes. Any suggestions?

Context: Gender: female, Age group: 20-30

Product search arguments:
category: 'shoes'
subcategory: 'boots'
color: 'black'



Input: Help me find something very simple for my first day at work next week. Something casual and neutral.

Context: Gender: male, Season: summer

Product search arguments:
category: 'tops'
subcategory: 't-shirts'
color: 'neutral'



Input: Help me find something very simpl

# PydanticAI

In [ ]:
!pip install -q pydantic-ai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.7/77.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.4/203.4 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.8/209.8 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.1/128.1 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.1/109.1 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 5.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.27.0, but you have google-auth 2.37.0 which is incompatible.


In [ ]:
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIModel

model = OpenAIModel('gpt-4o-2', openai_client=async_azure_client)
# Define a very simple agent including the model to use, you can also set the model when running the agent.
agent = Agent(
    model,
    # Register a static system prompt using a keyword argument to the agent.
    # For more complex dynamically-generated system prompts, see the example below.
    system_prompt='Be concise, reply with one sentence.',
)

result = agent.run_sync('Where does "hello world" come from?')

In [ ]:
result.data

'"Hello, World!" originated as a simple output example in the 1978 book "The C Programming Language" by Brian Kernighan and Dennis Ritchie.'

In [ ]:
from dataclasses import dataclass

from pydantic import BaseModel, Field

from pydantic_ai import Agent, RunContext


class DatabaseConn:
    """This is a fake database for example purposes.

    In reality, you'd be connecting to an external database
    (e.g. PostgreSQL) to get information about customers.
    """


    users = {
        'John Doe': {'id': 123, 'balance': 123.45},
        'Jane Doe': {'id': 456, 'balance': 678.90},
    }

    @classmethod
    async def customer_name(cls, *, id: int) -> str | None:
        if id == 123:
            return 'John'

    @classmethod
    async def customer_balance(cls, *, id: int, include_pending: bool) -> float:
        if id == 123:
            return 123.45
        else:
            raise ValueError('Customer not found')

In [ ]:
from dataclasses import dataclass

from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext

@dataclass
class SupportDependencies:
    customer_id: int
    db: DatabaseConn


class SupportResult(BaseModel):
    support_advice: str = Field(description='Advice returned to the customer')
    block_card: bool = Field(description="Whether to block the customer's card")
    risk: int = Field(description='Risk level of query', ge=0, le=10)


support_agent = Agent(
    model=model,
    deps_type=SupportDependencies,
    result_type=SupportResult,
    system_prompt=(
        'You are a support agent in our bank, give the '
        'customer support and judge the risk level of their query.'
    ),
)


@support_agent.system_prompt
async def add_customer_name(ctx: RunContext[SupportDependencies]) -> str:
    customer_name = await ctx.deps.db.customer_name(id=ctx.deps.customer_id)
    return f"The customer's name is {customer_name!r}"


@support_agent.tool
async def customer_balance(
    ctx: RunContext[SupportDependencies], include_pending: bool
) -> float:
    """Returns the customer's current account balance include name."""
    return await ctx.deps.db.customer_balance(
        id=ctx.deps.customer_id,
        include_pending=include_pending,
    )

In [ ]:
deps = SupportDependencies(customer_id=123, db=DatabaseConn())
result = support_agent.run_sync('What is my balance?', deps=deps)
print(result.data)

support_advice='Your current account balance, including any pending transactions, is $123.45. If you need any further assistance, feel free to ask!' block_card=False risk=0


In [ ]:
result = support_agent.run_sync('I just lost my card!', deps=deps)
print(result.data)

support_advice='We will take immediate steps to protect your account. Your card has been blocked to prevent unauthorized transactions. Please check your account balance for any unusual activity.' block_card=True risk=8


In [ ]:
c="Juan"
print(f"The customer's name is {c!a}") # ascii()
print(f"The customer's name is {c!r}") # repr()
print(f"The customer's name is {c!s}") # str()

The customer's name is 'Juan'
The customer's name is 'Juan'
The customer's name is Juan


In [ ]:
roulette_agent = Agent(
    model=model,
    deps_type=int,
    result_type=str,
    system_prompt=(
        'Use the `roulette_wheel` function to see if the '
        'customer has won based on the number they provide.'
    ),
)


@roulette_agent.tool
async def roulette_wheel(ctx: RunContext[int], square: int) -> str:
    """check if the square is a winner"""
    return 'winner' if square == ctx.deps else 'loser'

In [ ]:
success_number = 18
result = roulette_agent.run_sync('Put my money on square eighteen', deps=success_number)
print(result.data)

Congratulations! You've won by placing your bet on square eighteen.


In [ ]:
result = roulette_agent.run_sync('I bet five is the winner', deps=success_number)
print(result.data)

Unfortunately, five is not a winning number this time. Better luck next time!


In [ ]:
roulette_agent = Agent(
    model=model,
    deps_type=int,
    result_type=bool,
    system_prompt=(
        'Use the `roulette_wheel` function to see if the '
        'customer has won based on the number they provide.'
    ),
)


@roulette_agent.tool
async def roulette_wheel(ctx: RunContext[int], square: int) -> str:
    """check if the square is a winner"""
    return 'winner' if square == ctx.deps else 'loser'

In [ ]:
success_number = 18
result = roulette_agent.run_sync('Put my money on square eighteen', deps=success_number)
print(result.data)

True


In [ ]:
result = roulette_agent.run_sync('I bet five is the winner', deps=success_number)
print(result.data)

False


In [ ]:
agent = Agent(
    model, system_prompt='Be concise, reply with one word.',
)
result_sync = agent.run_sync('What is the capital of Italy?')
print(result_sync.data)

Rome.


In [ ]:
result = await agent.run('What is the capital of France?')
print(result.data)

Paris


In [ ]:
result.usage()

Usage(requests=1, request_tokens=26, response_tokens=1, total_tokens=27, details=None)

In [ ]:
async with agent.run_stream('What is the capital of the UK?') as response:
    print(await response.get_data(), end='')

London.


In [ ]:
agent = Agent(
    model
)

In [ ]:
result1 = agent.run_sync('How many starts exist in united state flag?')
print(result1.data)

The United States flag, also known as the Stars and Stripes, has 50 stars. Each star represents one of the 50 states in the United States.


In [ ]:
from IPython.display import clear_output
async with agent.run_stream('How many starts exist in united state flag? be extensive') as response:
    async for streamed_text in response.stream_text():
        clear_output(wait=True)
        print(streamed_text, end='')
    # print(response.stream_text(), end='')

The flag of the United States, commonly known as the Stars and Stripes, currently features 50 stars. Each star represents a state in the United States, reflecting the total number of states in the Union. 

Here's a more detailed look at the subject:

1. **Design**: The stars are white and set on a blue field, known as the "Union". This area is in the top left corner of the flag, called the canton. The stars are arranged in nine rows of alternating five or six stars, which offset each other.

2. **Historical Context**: The first official U.S. flag, established in 1777, had 13 stars to represent the original 13 colonies. As more states joined the Union, the number of stars on the flag increased accordingly. 

3. **Current Configuration**: The current 50-star version of the flag was adopted on July 4, 1960, following the admission of Hawaii as the 50th state on August 21, 1959.

4. **Symbolism**: Each star symbolizes a state and represents unity among the states. The blue field stands for

In [ ]:
result1.all_messages()

[ModelRequest(parts=[UserPromptPart(content='How many starts exist in united state flag?', timestamp=datetime.datetime(2024, 12, 30, 14, 7, 46, 819217, tzinfo=datetime.timezone.utc), part_kind='user-prompt')], kind='request'),
 ModelResponse(parts=[TextPart(content='The United States flag, also known as the Stars and Stripes, has 50 stars. Each star represents one of the 50 states in the United States.', part_kind='text')], timestamp=datetime.datetime(2024, 12, 30, 14, 7, 47, tzinfo=datetime.timezone.utc), kind='response')]

In [ ]:
result1.all_messages()

[ModelRequest(parts=[UserPromptPart(content='How many starts exist in united state flag?', timestamp=datetime.datetime(2024, 12, 30, 14, 7, 46, 819217, tzinfo=datetime.timezone.utc), part_kind='user-prompt')], kind='request'),
 ModelResponse(parts=[TextPart(content='The United States flag, also known as the Stars and Stripes, has 50 stars. Each star represents one of the 50 states in the United States.', part_kind='text')], timestamp=datetime.datetime(2024, 12, 30, 14, 7, 47, tzinfo=datetime.timezone.utc), kind='response')]

In [ ]:
result1 = agent.run_sync('Who was Albert Einstein?')
print(result1.data)

Albert Einstein was a theoretical physicist renowned for his groundbreaking contributions to the field of physics. He was born on March 14, 1879, in Ulm, in the Kingdom of Württemberg in the German Empire, and died on April 18, 1955, in Princeton, New Jersey, USA.

Einstein is most famous for developing the theory of relativity, particularly his equation \(E = mc^2\), which describes the equivalence of energy (E) and mass (m) with \(c\) being the speed of light in a vacuum. This equation is one of the most notable outcomes of his theory and has had profound implications for physics and our understanding of the universe.

In 1921, Einstein was awarded the Nobel Prize in Physics for his explanation of the photoelectric effect, which significantly advanced the understanding of quantum mechanics.

Beyond his scientific achievements, Einstein was an outspoken advocate for civil rights, pacifism, and Zionism. His insights and thoughts extended into philosophy and humanitarianism, making him 

In [ ]:
result1.all_messages()

[ModelRequest(parts=[UserPromptPart(content='Who was Albert Einstein?', timestamp=datetime.datetime(2024, 12, 30, 14, 21, 4, 373515, tzinfo=datetime.timezone.utc), part_kind='user-prompt')], kind='request'),
 ModelResponse(parts=[TextPart(content='Albert Einstein was a theoretical physicist renowned for his groundbreaking contributions to the field of physics. He was born on March 14, 1879, in Ulm, in the Kingdom of Württemberg in the German Empire, and died on April 18, 1955, in Princeton, New Jersey, USA.\n\nEinstein is most famous for developing the theory of relativity, particularly his equation \\(E = mc^2\\), which describes the equivalence of energy (E) and mass (m) with \\(c\\) being the speed of light in a vacuum. This equation is one of the most notable outcomes of his theory and has had profound implications for physics and our understanding of the universe.\n\nIn 1921, Einstein was awarded the Nobel Prize in Physics for his explanation of the photoelectric effect, which sig

In [ ]:
result1.new_messages()

[ModelRequest(parts=[UserPromptPart(content='Who was Albert Einstein?', timestamp=datetime.datetime(2024, 12, 30, 14, 21, 4, 373515, tzinfo=datetime.timezone.utc), part_kind='user-prompt')], kind='request'),
 ModelResponse(parts=[TextPart(content='Albert Einstein was a theoretical physicist renowned for his groundbreaking contributions to the field of physics. He was born on March 14, 1879, in Ulm, in the Kingdom of Württemberg in the German Empire, and died on April 18, 1955, in Princeton, New Jersey, USA.\n\nEinstein is most famous for developing the theory of relativity, particularly his equation \\(E = mc^2\\), which describes the equivalence of energy (E) and mass (m) with \\(c\\) being the speed of light in a vacuum. This equation is one of the most notable outcomes of his theory and has had profound implications for physics and our understanding of the universe.\n\nIn 1921, Einstein was awarded the Nobel Prize in Physics for his explanation of the photoelectric effect, which sig

In [ ]:
result2 = agent.run_sync(
    'What was his most famous equation?',
    message_history=result1.new_messages(),
)
print(result2.data)

Albert Einstein's most famous equation is \(E = mc^2\). This equation is a cornerstone of his theory of relativity and expresses the concept of mass-energy equivalence. In the equation:

- \(E\) stands for energy,
- \(m\) stands for mass, and
- \(c\) is the speed of light in a vacuum, approximately \(299,792,458\) meters per second.

The equation illustrates the principle that mass can be converted into energy and vice versa, implying that a small amount of mass can be converted into a large amount of energy due to the speed of light being a large number. This equation has fundamental implications in both theoretical physics and practical applications, including nuclear energy.


In [ ]:
%%writefile type_mistakes.py
from dataclasses import dataclass
from pydantic_ai import Agent, RunContext

@dataclass
class User:
    name: str


agent = Agent(
    'test',
    deps_type=User,
    result_type=bool,
)


@agent.system_prompt
def add_user_name(ctx: RunContext[str]) -> str:
    return f"The user's name is {ctx.deps.name}."


def foobar(x: bytes) -> None:
    pass


result = agent.run_sync('Does their name start with "A"?', deps=User('Anne'))
foobar(result.data)

Writing type_mistakes.py


In [ ]:
!uv run -p python type_mistakes.py

In [ ]:
from datetime import date

from pydantic_ai import Agent, RunContext

agent = Agent(
    model=model,
    deps_type=str,
    system_prompt="Use the customer's name while replying to them.",
)


@agent.system_prompt
def add_the_users_name(ctx: RunContext[str]) -> str:
    return f"The user's named is {ctx.deps}."


@agent.system_prompt
def add_the_date() -> str:
    return f'The date is {date.today()}.'


result = agent.run_sync('What is the date?', deps='Frank')
print(result.data)

Today's date is December 30, 2024, Frank.


In [ ]:
import random

from pydantic_ai import Agent, RunContext

agent = Agent(
    model=model,
    deps_type=str,
    system_prompt=(
        "You're a dice game, you should roll the die and see if the number "
        "you get back matches the user's guess. If so, tell them they're a winner. "
        "Use the player's name in the response."
    ),
)


@agent.tool_plain
def roll_die() -> str:
    """Roll a six-sided die and return the result."""
    print('Calling the `roll_die` tool...')
    return str(random.randint(1, 6))


@agent.tool
def get_player_name(ctx: RunContext[str]) -> str:
    """Get the player's name."""
    print('Calling the `get_player_name` tool...')
    return ctx.deps


dice_result = agent.run_sync('My guess is 4', deps='Anne')
print(dice_result.data)

Calling the `roll_die` tool...
Calling the `get_player_name` tool...
Sorry, Anne, the roll was a 3. Better luck next time!


In [ ]:
from pprint import pprint
pprint(dice_result.all_messages())

[ModelRequest(parts=[SystemPromptPart(content="You're a dice game, you should "
                                              'roll the die and see if the '
                                              'number you get back matches the '
                                              "user's guess. If so, tell them "
                                              "they're a winner. Use the "
                                              "player's name in the response.",
                                      part_kind='system-prompt'),
                     UserPromptPart(content='My guess is 4',
                                    timestamp=datetime.datetime(2024, 12, 30, 14, 23, 34, 627532, tzinfo=datetime.timezone.utc),
                                    part_kind='user-prompt')],
              kind='request'),
 ModelResponse(parts=[ToolCallPart(tool_name='roll_die',
                                   args=ArgsJson(args_json='{}'),
                                   tool_call_id='call_

In [ ]:
dice_result = agent.run_sync('My guess is one', deps='Anne')
print(dice_result.data)

Calling the `roll_die` tool...Calling the `get_player_name` tool...

Congratulations, Anne! You guessed correctly. You're a winner! 🎉


In [ ]:
from pprint import pprint
pprint(dice_result.all_messages())

[ModelRequest(parts=[SystemPromptPart(content="You're a dice game, you should "
                                              'roll the die and see if the '
                                              'number you get back matches the '
                                              "user's guess. If so, tell them "
                                              "they're a winner. Use the "
                                              "player's name in the response.",
                                      part_kind='system-prompt'),
                     UserPromptPart(content='My guess is one',
                                    timestamp=datetime.datetime(2024, 12, 30, 14, 27, 30, 306131, tzinfo=datetime.timezone.utc),
                                    part_kind='user-prompt')],
              kind='request'),
 ModelResponse(parts=[ToolCallPart(tool_name='roll_die',
                                   args=ArgsJson(args_json='{}'),
                                   tool_call_id='cal

In [ ]:
import random

from pydantic_ai import Agent, RunContext, Tool


def roll_die() -> str:
    """Roll a six-sided die and return the result."""
    return str(random.randint(1, 6))


def get_player_name(ctx: RunContext[str]) -> str:
    """Get the player's name."""
    return ctx.deps


agent_a = Agent(
    model=model,
    deps_type=str,
    tools=[roll_die, get_player_name],
)
agent_b = Agent(
    model=model,
    deps_type=str,
    tools=[
        Tool(roll_die, takes_ctx=False),
        Tool(get_player_name, takes_ctx=True),
    ],
)
dice_result = agent_b.run_sync('My guess is 4', deps='Anne')
print(dice_result.data)

The result of the die roll is 4, and the player's name is Anne. Great guess, Anne!


In [ ]:
from pprint import pprint
pprint(dice_result.all_messages())

[ModelRequest(parts=[UserPromptPart(content='My guess is 4',
                                    timestamp=datetime.datetime(2024, 12, 30, 14, 28, 54, 516356, tzinfo=datetime.timezone.utc),
                                    part_kind='user-prompt')],
              kind='request'),
 ModelResponse(parts=[ToolCallPart(tool_name='roll_die',
                                   args=ArgsJson(args_json='{}'),
                                   tool_call_id='call_yNWouWLGsxR70YMVYjq8jyxW',
                                   part_kind='tool-call'),
                      ToolCallPart(tool_name='get_player_name',
                                   args=ArgsJson(args_json='{}'),
                                   tool_call_id='call_Wmz80J5jIVwvLiyjV9rD3sUY',
                                   part_kind='tool-call')],
               timestamp=datetime.datetime(2024, 12, 30, 14, 28, 54, tzinfo=datetime.timezone.utc),
               kind='response'),
 ModelRequest(parts=[ToolReturnPart(tool_name='ro

In [ ]:
dice_result = agent_a.run_sync('My guess is 4', deps='Anne')
print(dice_result.data)

Hi Anne! You guessed 4, but the die roll was actually a 5. Better luck next time!


In [ ]:
from pprint import pprint
pprint(dice_result.all_messages())

[ModelRequest(parts=[UserPromptPart(content='My guess is 4',
                                    timestamp=datetime.datetime(2024, 12, 30, 14, 29, 28, 996808, tzinfo=datetime.timezone.utc),
                                    part_kind='user-prompt')],
              kind='request'),
 ModelResponse(parts=[ToolCallPart(tool_name='get_player_name',
                                   args=ArgsJson(args_json='{}'),
                                   tool_call_id='call_QlFgLnCU6QtJC2gxOJ7fNDDH',
                                   part_kind='tool-call'),
                      ToolCallPart(tool_name='roll_die',
                                   args=ArgsJson(args_json='{}'),
                                   tool_call_id='call_KaHXrL2e0FnjG6zOA9ifBfzN',
                                   part_kind='tool-call')],
               timestamp=datetime.datetime(2024, 12, 30, 14, 29, 29, tzinfo=datetime.timezone.utc),
               kind='response'),
 ModelRequest(parts=[ToolReturnPart(tool_name='ge

In [ ]:
from pydantic_ai import Agent
from pydantic_ai.messages import ModelMessage, ModelResponse
from pydantic_ai.models.function import AgentInfo, FunctionModel

agent = Agent(model=model)


@agent.tool_plain
def foobar(a: int, b: str, c: dict[str, list[float]]) -> str:
    """Get me foobar.

    Args:
        a: apple pie
        b: banana cake
        c: carrot smoothie
    """
    return f'{a} {b} {c}'


def print_schema(messages: list[ModelMessage], info: AgentInfo) -> ModelResponse:
    idx_fun_names = list(map(lambda x: x.name, info.function_tools))
    idx = idx_fun_names.index('foobar')
    tool = info.function_tools[idx]
    pprint(tool.description)
    pprint(tool.parameters_json_schema)
    #> Get me foobar.
    return ModelResponse.from_text(content='foobar')


agent.run_sync('hello', model=FunctionModel(print_schema))

'Get me foobar.'
{'additionalProperties': False,
 'properties': {'a': {'description': 'apple pie',
                      'title': 'A',
                      'type': 'integer'},
                'b': {'description': 'banana cake',
                      'title': 'B',
                      'type': 'string'},
                'c': {'additionalProperties': {'items': {'type': 'number'},
                                               'type': 'array'},
                      'description': 'carrot smoothie',
                      'title': 'C',
                      'type': 'object'}},
 'required': ['a', 'b', 'c'],
 'type': 'object'}


RunResult(_all_messages=[ModelRequest(parts=[UserPromptPart(content='hello', timestamp=datetime.datetime(2024, 12, 30, 14, 54, 12, 967142, tzinfo=datetime.timezone.utc), part_kind='user-prompt')], kind='request'), ModelResponse(parts=[TextPart(content='foobar', part_kind='text')], timestamp=datetime.datetime(2024, 12, 30, 14, 54, 12, 982166, tzinfo=datetime.timezone.utc), kind='response')], _new_message_index=0, data='foobar', _usage=Usage(requests=1, request_tokens=51, response_tokens=1, total_tokens=52, details=None))

In [ ]:
from pydantic import BaseModel

from pydantic_ai import Agent, RunContext, ModelRetry


class ChatResult(BaseModel):
    user_id: int
    message: str

agent = Agent(
    model=model,
    deps_type=DatabaseConn,
    result_type=ChatResult,
)


@agent.tool(retries=4)
def get_user_by_name(ctx: RunContext[DatabaseConn], name: str, cup_type: str) -> int:
    """Get a user's ID from their full name."""
    print(name, cup_type)
    user_id = (ctx.deps.users.get(name,None) or {}).get('id',None)
    if user_id is None:
        raise ModelRetry(
            f'No user found with name {name!r}, remember to provide their full name'
        )
    return user_id


result = agent.run_sync(
    'Send a message to John D.  asking for cofe next week', deps=DatabaseConn()
)
print(result.data)

John D. cofe
John Doe cofe
user_id=123 message='Would you like to grab coffee together sometime next week?'


In [ ]:
result = agent.run_sync(
    'Send a message to David D. asking for coffee next week', deps=DatabaseConn()
)
print(result.data)

David D. coffee
David D coffee
David coffee
user_id=0 message='I\'m unable to find a user by the name "David D.". Please check the name and try again with the full, correct name.'


In [ ]:
result = agent.run_sync(
    'Send a message to J.D. asking for coffee next week', deps=DatabaseConn()
)
print(result.data)

J.D. user_id
John Doe user_id
user_id=123 message='Hey John, would you like to grab coffee sometime next week?'


In [ ]:
from pydantic_ai import Agent, ModelRetry, UnexpectedModelBehavior

agent = Agent(model=model)


@agent.tool_plain
def calc_volume(size: int) -> int:  # (1)!
    if size == 42:
        return size**3
    else:
        raise ModelRetry('Please try again.')


try:
    result = agent.run_sync('Please get me the volume of a box with size 6.')
except UnexpectedModelBehavior as e:
    print('An error occurred:', e)
    #> An error occurred: Tool exceeded max retries count of 1
    print('cause:', repr(e.__cause__))
    #> cause: ModelRetry('Please try again.')
    print('messages:', agent.last_run_messages)
else:
    print(result.data)

An error occurred: Tool exceeded max retries count of 1
cause: ModelRetry('Please try again.')
messages: [UserPrompt(content='Please get me the volume of a box with size 6.', timestamp=datetime.datetime(2024, 12, 6, 17, 15, 29, 294382, tzinfo=datetime.timezone.utc), role='user'), ModelStructuredResponse(calls=[ToolCall(tool_name='calc_volume', args=ArgsJson(args_json='{"size":6}'), tool_id='call_sEUrsieVJEUCwgnVpyfKCSXZ')], timestamp=datetime.datetime(2024, 12, 6, 17, 15, 29, tzinfo=datetime.timezone.utc), role='model-structured-response'), RetryPrompt(content='Please try again.', tool_name='calc_volume', tool_id='call_sEUrsieVJEUCwgnVpyfKCSXZ', timestamp=datetime.datetime(2024, 12, 6, 17, 15, 29, 773529, tzinfo=datetime.timezone.utc), role='retry-prompt'), ModelStructuredResponse(calls=[ToolCall(tool_name='calc_volume', args=ArgsJson(args_json='{"size":6}'), tool_id='call_EWB2fYayrfrWj9paibyzeMB0')], timestamp=datetime.datetime(2024, 12, 6, 17, 15, 29, tzinfo=datetime.timezone.utc), r

In [ ]:
!pip install git+https://github.com/openai/swarm.git -qq

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.1/70.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.7/218.7 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.0/99.0 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.2/325.2 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 82.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.0/469.0 kB 29.5 MB/s eta 0:00:00


In [ ]:
from swarm import Swarm, Agent

client = Swarm(azure_client)

In [ ]:
from swarm import Swarm, Agent

client = Swarm(azure_client)

# Define language agents
english_agent = Agent(
    name="English Support Agent",
    instructions="You handle inquiries only in English.",
)

spanish_agent = Agent(
    name="Spanish Support Agent",
    instructions="Respondes preguntas únicamente en español.",
)

# Define tiered support agents
basic_support_agent = Agent(
    name="Basic Support Agent",
    instructions="Provide basic troubleshooting for common issues.",
)

advanced_support_agent = Agent(
    name="Advanced Support Agent",
    instructions="Handle advanced technical issues requiring detailed expertise using bullet points with two levels.",
)

# Define transfer functions
def transfer_to_spanish_agent():
    """Detect and transfer Spanish-speaking users to the Spanish agent."""
    return spanish_agent

def escalate_to_advanced_support():
    """Escalate complex queries to the advanced technical/settings support agent."""
    return advanced_support_agent

# Add transfer functions to appropriate agents
english_agent.functions.append(transfer_to_spanish_agent)
basic_support_agent.functions.append(escalate_to_advanced_support)

# english_agent.functions.extend([transfer_to_spanish_agent, basic_support_agent, escalate_to_advanced_support])
# TypeError: Agent(name='Basic Support Agent', model='gpt-4o', instructions='Provide basic troubleshooting for common issues.', functions=[], tool_choice=None, parallel_tool_calls=True) is not a callable object

# Example message pipeline
messages = [
    {"role": "user", "content": "Hola, mi internet no funciona."},  # Spanish message
    {"role": "user", "content": "I need help with advanced settings"},  # Advanced query
    {"role": "user", "content": "Traducemelo lo último al español"},  # Translating query
]

# Initial agent
starting_agent = english_agent
historical_messages = []

for i, message in enumerate(messages):
    print("-"*100)
    print(f"Processing message: {message['content']}")

    historical_messages.append(message)
    response = client.run(agent=starting_agent, messages=historical_messages)
    reply = response.messages[-1]["content"]
    historical_messages.extend(response.messages)

    print(f"Agent ({starting_agent.name}) response: {reply}")

    # Check if the agent returned a transfer
    if response_agent := response.messages[-1]:
        print(f"Transferring to: {response_agent['sender']}")
    else:
        # End processing if no further transfer
        print("End of message handling.")
        break
    print("-"*100)


----------------------------------------------------------------------------------------------------
Processing message: Hola, mi internet no funciona.
Agent (English Support Agent) response: ¡Hola! Lamento que tengas problemas con tu conexión a internet. Para poder ayudarte mejor, necesito saber algunos detalles más:

1. ¿Qué tipo de conexión a internet tienes (wifi, cable, etc.)?
2. ¿Has intentado reiniciar tu módem o router?
3. ¿Qué luces están encendidas en tu módem o router?
4. ¿Estás experimentando problemas en todos los dispositivos o solo en uno?

Con esta información podré darte una mejor solución.
Transferring to: Spanish Support Agent
----------------------------------------------------------------------------------------------------
----------------------------------------------------------------------------------------------------
Processing message: I need help with advanced settings
Agent (English Support Agent) response: Could you please specify which advanced settings 